In [0]:
CREATE OR REPLACE VIEW 03_dev_gold.kpi_tables.overall_view AS
SELECT
    ROUND(COUNT(DISTINCT order_id),2) as total_order_count,
    ROUND(SUM(base_line_total),2) AS total_revenue,
    COUNT(DISTINCT CASE WHEN order_status = 'COMPLETED' THEN order_id END) AS completed_order_count,
    ROUND(COUNT(DISTINCT CASE WHEN order_status = 'COMPLETED' THEN order_id END)/COUNT(DISTINCT order_id),2) AS completed_order_rate,
    ROUND(SUM(base_line_total)  / COUNT(DISTINCT order_id),2) AS average_order_value,
    COUNT(DISTINCT customer_key) AS active_customers
FROM 02_dev_silver.facts_and_dims.fact_order_items;


CREATE OR REPLACE VIEW 03_dev_gold.kpi_tables.by_country_by_channel AS
SELECT
    'country' AS country_or_channel,
    dc.country AS dimension_value,
    ROUND(SUM(f.base_line_total),2) AS revenue
FROM 02_dev_silver.facts_and_dims.fact_order_items f
LEFT JOIN 02_dev_silver.facts_and_dims.dim_customer dc
    ON f.customer_key = dc.customer_key
GROUP BY dc.country

UNION ALL

SELECT
    'channel' AS country_or_channel,
    dc.channel AS dimension_value,
    ROUND(SUM(f.base_line_total),2) AS revenue
FROM 02_dev_silver.facts_and_dims.fact_order_items f
LEFT JOIN 02_dev_silver.facts_and_dims.dim_customer dc
    ON f.customer_key = dc.customer_key
GROUP BY dc.channel
ORDER BY dimension_value;


CREATE OR REPLACE VIEW 03_dev_gold.kpi_tables.top_5_products AS
SELECT *
FROM 
(
    SELECT
        dp.product_name,
        ROUND(SUM(f.base_line_total),2) AS revenue,
        RANK() OVER (ORDER BY SUM(f.base_line_total) DESC) AS rank
    FROM 02_dev_silver.facts_and_dims.fact_order_items f
    LEFT JOIN 02_dev_silver.facts_and_dims.dim_product dp
        ON f.product_key = dp.product_key
    GROUP BY dp.product_name
)
WHERE rank <= 5;


CREATE OR REPLACE VIEW 03_dev_gold.kpi_tables.customer_acquistion AS
SELECT
    YEAR(full_date) AS year,
    MONTH(full_date) AS month,
    COUNT(DISTINCT customer_key) AS customers_acquired
FROM 02_dev_silver.facts_and_dims.dim_customer dc
LEFT JOIN 02_dev_silver.facts_and_dims.dim_date dd
ON dc.registration_date_key = dd.date_key
WHERE full_date IS NOT NULL
GROUP BY YEAR(full_date), MONTH(full_date)
ORDER BY year, month;